In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

spark = SparkSession.builder \
    .appName("Hybrid_Model_Research") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

In [2]:
# Đọc kết quả đã lưu
als_recs = spark.table("gold_db.recommendations_als") # user_id, product_id, score
cb_recs = spark.table("gold_db.recommendations_content_based") # product_id_1 (as product_id), product_id_2, content_score

# Chuẩn hóa điểm về [0, 1] để cộng lại cho công bằng (Min-Max Scaling)
def normalize(df, score_col):
    stats = df.select(F.min(score_col).alias("min"), F.max(score_col).alias("max")).first()
    return df.withColumn("norm_score", (F.col(score_col) - stats["min"]) / (stats["max"] - stats["min"]))

als_norm = normalize(als_recs, "score").select("user_id", "product_id", F.col("norm_score").alias("als_score"))
# Với CB, ta lấy gợi ý dựa trên lịch sử tương tác 
cb_recs_final = spark.table("gold_db.recommendations_content_based") \
    .select(F.col("product_id_1").alias("source_p"), F.col("product_id_2").alias("product_id"), F.col("content_score"))

# Lấy lịch sử tương tác của User để lai
user_history = spark.table("gold_db.user_interactions").select("user_id", F.col("product_id").alias("source_p"))

cb_final = user_history.join(cb_recs_final, "source_p") \
    .groupBy("user_id", "product_id").agg(F.max("content_score").alias("cb_raw_score"))

cb_norm = normalize(cb_final, "cb_raw_score").select("user_id", "product_id", F.col("norm_score").alias("cb_score"))

In [7]:
import datetime

# TẠO BẢNG CƠ SỞ (dữ liệu ALS và Content-Based)
combined_base = als_norm.join(cb_norm, ["user_id", "product_id"], "outer").fillna(0)
combined_base = combined_base.localCheckpoint() # Lưu tạm vào disk của container để giải phóng RAM

# CHUẨN BỊ DỮ LIỆU TEST 
events_df = spark.table("gold_db.fact_events")
max_date = events_df.select(F.max("event_time")).first()[0]
cutoff = max_date - datetime.timedelta(days=30)
test_matrix = events_df.filter(F.col("event_time") >= cutoff) \
    .groupBy("user_id").agg(F.collect_set("product_id").alias("actual_list")).cache()

#  HÀM ĐÁNH GIÁ 
def evaluate_hybrid_fast(df_weighted, test_data_cached, k=10):
    window_spec = Window.partitionBy("user_id").orderBy(F.desc("hybrid_score"))
    # Chỉ lấy top K ngay lập tức để giảm dữ liệu
    top_k = df_weighted.withColumn("rank", F.row_number().over(window_spec)).filter(F.col("rank") <= k)
    
    predicted = top_k.groupBy("user_id").agg(F.collect_set("product_id").alias("pred_list"))
    
    # Join với bảng thực tế đã được cache
    comparison = test_data_cached.join(predicted, "user_id")
    match_df = comparison.withColumn("hits", F.size(F.array_intersect("actual_list", "pred_list")))
    
    # Tính toán chỉ số
    results = match_df.select(
        F.avg(F.col("hits") / k).alias("precision"),
        F.avg(F.col("hits") / F.size("actual_list")).alias("recall")
    ).first()
    
    p = results['precision']
    r = results['recall']
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    return p, r, f1

#  VÒNG LẶP TÌM TRỌNG SỐ 
weight_options = [(0.9, 0.1), (0.8, 0.2), (0.7, 0.3), (0.5, 0.5)]
best_f1 = 0
best_w = (0, 0)

for w_als, w_cb in weight_options:
    temp_hybrid = combined_base.withColumn(
        "hybrid_score", (F.col("als_score") * w_als) + (F.col("cb_score") * w_cb)
    )
    p, r, f = evaluate_hybrid_fast(temp_hybrid, test_matrix)
    print(f"-> ALS={w_als}, CB={w_cb} | F1: {f:.4f} | Prec: {p:.4f}")
    
    if f > best_f1:
        best_f1 = f
        best_w = (w_als, w_cb)

print(f"\nTRỌNG SỐ TỐI ƯU: ALS={best_w[0]}, CB={best_w[1]} (F1={best_f1:.4f})")

-> ALS=0.9, CB=0.1 | F1: 0.0078 | Prec: 0.0047
-> ALS=0.8, CB=0.2 | F1: 0.0063 | Prec: 0.0038
-> ALS=0.7, CB=0.3 | F1: 0.0061 | Prec: 0.0037
-> ALS=0.5, CB=0.5 | F1: 0.0058 | Prec: 0.0036

TRỌNG SỐ TỐI ƯU: ALS=0.9, CB=0.1 (F1=0.0078)


In [ ]:
# Chọn mô hình ALS hoặc Hybrid tùy m (t khuyên dùng ALS cho khách cũ vì F1 cao nhất)
# Ở đây t hướng dẫn lưu bảng Hybrid để m khoe tính năng Lai với thầy

# 1. Lấy Top 10 gợi ý Hybrid cho mỗi User
window_final = Window.partitionBy("user_id").orderBy(F.desc("score"))
final_recommendations = final_top_10.select("user_id", "product_id", "score")

# 2. Đẩy sang PostgreSQL (Serving Layer)
# M cần chắc chắn Postgres container đang chạy
print("Đang đẩy kết quả gợi ý sang PostgreSQL...")
final_recommendations.write.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/data_lakehouse") \
    .option("dbtable", "gold_db.final_recommendations") \
    .option("user", "user") \
    .option("password", "password") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

print("--- THÀNH CÔNG: Dữ liệu đã sẵn sàng cho Web Demo! ---")